# Build ecmtool inputs

Prepares the Rhodoferax and Geobacter models for ecmtool: for each species and each exchange pattern, fixes a growth rate and carbon-uptake rate, converts the model into a bound-free form ecmtool can read, and writes out one SBML file per species/pattern/growth-rate combination plus the shell script to run ecmtool on all of them.

1. Load the model, restrict exchanges to the pattern's allowed uptakes/secretions, close all others.
2. Determine the maximum growth rate under this pattern.
3. For a set of growth-rate fractions (0.3-0.99 of the max), fix growth rate and carbon uptake:
    * Add a `fixed_growth` metabolite, produced by the biomass reaction.
    * Add a `fixed_uptake` metabolite, consumed by the carbon-source uptake reaction.
    * Add `constraint_reaction`, with flux fixed at 1, consuming both metabolites at rates that encode the target growth rate and uptake. This converts growth rate and uptake into equalities, since ecmtool cannot read bounds from the SBML file - it treats every row as an equality.
    * Add `slack_reaction` so the uptake constraint stays an inequality (uptake <= `FIXED_UPTAKE`) instead of becoming an equality too.
4. Export one SBML file per species/pattern/growth-rate combination.
5. Find the `--tag` index ecmtool needs for `constraint_reaction`, and write the `.sh` file that runs ecmtool on all generated models.
6. Save everything a downstream notebook needs (growth rates, patterns, tags) to `run_metadata.pkl`, so it doesn't have to be copy-pasted.

In [1]:
import os
import sys
import pickle
import contextlib
from collections import defaultdict

import cobra
import numpy as np
from cobra import Reaction

from helper import flip_reverse_reactions, add_product_to_reaction

sys.path.insert(0, "/home/users/dszeliova/ecmtool")
from ecmtool.network import extract_sbml_stoichiometry

cobra_config = cobra.Configuration()
cobra_config.solver = "cplex"
cobra_config.tolerance = 1e-9

# Maximum carbon-source uptake rate enforced everywhere below (FBA bounds and
# the ecmtool constraint trick). Defined once here and saved to run_metadata.pkl
# at the end, so analyze_ecms.ipynb reads it instead of hardcoding it again.
FIXED_UPTAKE = 10

/home/users/dszeliova/ecmtool/ecmtool/_bglu_dense.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import sys, pkg_resources, imp


### Define exchange patterns

Selected exchange patterns from Oftadeh et al. 2024: https://doi.org/10.1101/2024.01.30.577913

In [2]:
bm_rxns = {"Geobacter": "agg_GS13m", "Rhodoferax": "BIO_Rfer3"}
c_sources = ["EX_mal-L_e",  "EX_fum_e"] #"EX_cit_e",


exchange_pattern = {"Rhodoferax": {
            "mEmC_1": {"uptakes": ["EX_fe2_e", "EX_fum_e"],
                        "secretion": ["EX_mal-L_e"]},
            # "mEmC_3": {"uptakes": [ "EX_fe2_e", "EX_fum_e"],
            #             "secretion": ["EX_mal-L_e"]},
            # "mEmC_2": {"uptakes": [ "EX_fe2_e", "EX_cit_e"],
            #           "secretion": ["EX_mal-L_e"]},
            # "mEmC_4": {"uptakes": [ "EX_fe2_e", "EX_cit_e"],
            #             "secretion": ["EX_mal-L_e"]}
},
        "Geobacter": {
            "mEmC_1": {"uptakes": ["EX_nh4_e", "EX_mal-L_e"],
                      "secretion": []}, #"EX_co2_e"
            # "mEmC_2": {"uptakes": ["EX_nh4_e", "EX_mal-L_e"],
            #           "secretion": []},
            # "mEmC_3": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
            #           "secretion": []},
            # "mEmC_4": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
            #           "secretion": []}
    }}

# from their github
def open_exchanges(model, organism):

    if organism == 'Geobacter':
        allowed = ['so4_e', 'pi_e', 'mg2_e', 'k_e', 'ca2_e', 'fe3_e']
    elif organism == 'Rhodoferax':
        allowed = ['nh4_e', 'pi_e', 'so4_e', 'o2_e']
    else:
        raise ValueError(f"Unknown organism: {organism}")

    # remove the demand reactions in Geobacter
    if organism == "Geobacter":
        model.reactions.ATPM.bounds = (0., 1000.)

        to_remove = [rxn.id for rxn in model.reactions if rxn.id.startswith("DM_")]
        model.remove_reactions(to_remove)

    return allowed


def implement_pattern(model, allowed, pattern, c_sources, allow_secretions=True):
    # first, allow all exchanges as in the paper and turn off all secretions
    mu = model.slim_optimize()
    print(f" * Initial growth rate: {mu:.2f}")

    to_remove = []
    for ex in model.exchanges:
        if ex.id in [ "EX_h2o_e",  "EX_h2_e",  "EX_h_e" ]:
            ex.bounds = (-1000, 1000)
            continue

        # pattern should overwrite the allowed
        if ex.id in pattern["secretion"]:
            ex.bounds = (0, 1000)
        elif (ex.id.replace("EX_", "") in allowed) or (ex.id in pattern["uptakes"]):
            ex.bounds = (-1000, 0)
            if ex.id in c_sources:
                ex.bounds = (-FIXED_UPTAKE, 0)
        else:
            ex.bounds = (0, 1000)
            if not allow_secretions:
                to_remove.append(ex)

    model.remove_reactions(to_remove)
    cobra.manipulation.delete.prune_unused_metabolites(model)

    mu = model.slim_optimize()
    print(f" * Growth rate with exchange pattern: {mu:.2f}")

    return mu

### Implement exchange pattern

Define uptakes according to the pattern and remove reactions with lower and upper bounds of zero. Test growth after implementing the constraints.

In [11]:
for modelname in reduced_models:
    print(f"\n***** {modelname} *****")

    for pattern_name, pattern in exchange_pattern[modelname].items():
        print(f"Testing exchange pattern {pattern_name}")

        model = cobra.io.read_sbml_model(f"../models/xml_models/{modelname}.xml")
        allowed = open_exchanges(model, modelname)

        if modelname == "Rhodoferax":
            fe2 = model.metabolites.get_by_id("fe2_c")
            rxn = model.reactions.get_by_id("FCLT")
            rxn.add_metabolites({fe2: -200000. }, combine=False)

            fe3 = model.metabolites.get_by_id("fe3_e")
            rxn = model.reactions.get_by_id("FERCYT")
            rxn.add_metabolites({fe3: -1000.}, combine=False)

            # needed for pycomo
            if pattern_name == "mEmC_1":
                cobra.io.write_sbml_model(model, f"../models/xml_models_test/{modelname}_heme.xml")
        


***** Geobacter *****
Testing exchange pattern mEmC_1
 * Initial growth rate: 35.65
 * Growth rate with exchange pattern: 1.02

***** Rhodoferax *****
Testing exchange pattern mEmC_1
 * Initial growth rate: 39.26
 * Growth rate with exchange pattern: 0.67


In [4]:
model.metabolites.fe2_e

Metabolite identifier,fe2_e
Name,fe2_e
Memory address,0x7fe0314e9a90
Formula,Fe
Compartment,e
In 3 reaction(s),"FE2abc, EX_fe2_e, FERCYT"


In [5]:
model.reactions.FERCYT

Reaction identifier,FERCYT
Name,Fe (III) Reductase: G sulfurreducens
Memory address,0x7fe0312c8090
Stoichiometry,500.0 fe3_e + focytcc_c --> fe2_e + ficytcc_c 500.0 fe3_e + focytcc_c --> fe2_e + ficytcc_c
GPR,
Lower bound,0.0
Upper bound,1000.0
